<a href="https://colab.research.google.com/github/UlaStats/MSc-project-pipe-failure-prediction/blob/main/Data_Preparation_RNN_and_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSc Project - Data Preparation for LSTM

This notebook contains code for and MSc Project about predicting time-to-failure of water mains in Scotland. In particular, the code contains data preparation for the LSTM model - cleaning and manipulation.

## Import Data

Data has already been cleaned and joined and the resultant data is imported into this notebook.

In [1]:
# mount Google Drive

from google.colab import drive

drive.mount("/content/drive", force_remount = True)

Mounted at /content/drive


In [2]:
# import data

import pandas as pd

data_cleaned_joined = pd.read_csv("/content/drive/MyDrive/MSc project/data_cleaned_joined.csv", encoding = "latin1")

In [3]:
# transform commissioned date to date type

data_cleaned_joined['Date_commissioned'] = pd.to_datetime(data_cleaned_joined['Date_commissioned'])

data_cleaned_joined['Raised.Date'] = pd.to_datetime(data_cleaned_joined['Raised.Date'])

## Data Manipulation

"Previous bursts" column is added.

In [4]:
total_bursts = data_cleaned_joined['Asset.ID'].value_counts().reset_index()

data_cleaned_joined = data_cleaned_joined.sort_values(["Asset.ID", "Raised.Date"])

for asset in data_cleaned_joined['Asset.ID']:

  previous_bursts = total_bursts['count'][total_bursts['Asset.ID'] == asset]

  data_cleaned_joined.loc[data_cleaned_joined['Asset.ID'] == asset, "Previous bursts"] = list(range(previous_bursts.iloc[0]))




Attribute "Time-to-failure" is added.

In [27]:
start_date = data_cleaned_joined['Raised.Date'].min()
data_cleaned_joined_engineered = []


for asset, bursts in data_cleaned_joined.groupby("Asset.ID"):

  for _,row in bursts.iterrows():

    if(row["Previous bursts"] == 0 and row["Date_commissioned"] > start_date):

      row["Time-to-failure"] = row["Raised.Date"] - row["Date_commissioned"]




    elif(row['Previous bursts'] == 0):

      row["Time-to-failure"] = row["Raised.Date"] - start_date


    else:

      row['Time-to-failure'] = row['Raised.Date'] - prev_row["Raised.Date"]

    data_cleaned_joined_engineered.append(row)
    prev_row = row

data_cleaned_joined_engineered = pd.DataFrame(data_cleaned_joined_engineered)

Column "Age" is added.

In [28]:
data_cleaned_joined_engineered_2 = []


for asset, bursts in data_cleaned_joined_engineered.groupby("Asset.ID"):

  for _,row in bursts.iterrows():

    if(row['Previous bursts'] == 0 and row["Date_commissioned"] > start_date):

      row["Age"] = start_date - start_date # this is used to obtain 0 days (0 in data type date)


    elif(row["Previous bursts"] == 0):

      row["Age"] = start_date - row["Date_commissioned"]


    else:

      row['Age'] = prev_row["Age"] + prev_row["Time-to-failure"]

    data_cleaned_joined_engineered_2.append(row)
    prev_row = row

data_cleaned_joined_engineered_2 = pd.DataFrame(data_cleaned_joined_engineered_2)

In [29]:
data_cleaned_joined_engineered_2['Time-to-failure'] = data_cleaned_joined_engineered_2["Time-to-failure"].dt.days/365

In [31]:
data_cleaned_joined_engineered_2["Age"] = data_cleaned_joined_engineered_2["Age"].dt.days/365

In [38]:
data_cleaned_joined_engineered_2["Raised.Date"]

,Raised.Date
7025,2019-11-21
22078,2009-12-02
22079,2010-11-15
22080,2011-11-02
22081,2013-02-05
...,...
7024,2011-05-18
19789,2006-07-21
19790,2017-12-15
20267,2011-10-06


In [40]:
attributes_train = data_cleaned_joined_engineered_2[data_cleaned_joined_engineered_2['Raised.Date'] <= "2019-12-31"]

In [43]:
attributes_valid = data_cleaned_joined_engineered_2[(data_cleaned_joined_engineered_2['Raised.Date'] >= "2020-01-01") & (data_cleaned_joined_engineered_2['Raised.Date'] <= "2021-12-31")]

In [44]:
attributes_test = data_cleaned_joined_engineered_2[data_cleaned_joined_engineered_2['Raised.Date'] >= "2022-01-01"]

In [61]:
X_train_sequential = attributes_train[["Asset.ID", "Material", "Diameter", "Length", "Lining", "Surface", "Soil", "Soil_pH", "Frost_days", "Hydrogen Ion", "Free chlorine", "Previous bursts", "Age"]]
y_train_sequential = attributes_train['Time-to-failure']

In [62]:
X_test_sequential = attributes_test[["Asset.ID", "Material", "Diameter", "Length", "Lining", "Surface", "Soil", "Soil_pH", "Frost_days", "Hydrogen Ion", "Free chlorine", "Previous bursts", "Age"]]
y_test_sequential = attributes_test['Time-to-failure']

In [63]:
X_valid_sequential = attributes_valid[["Asset.ID", "Material", "Diameter", "Length", "Lining", "Surface", "Soil", "Soil_pH", "Frost_days", "Hydrogen Ion", "Free chlorine", "Previous bursts", "Age"]]
y_valid_sequential = attributes_valid['Time-to-failure']

In [64]:
X_train_sequential.to_csv("/content/drive/MyDrive/MSc project/X_train_sequential.csv", encoding ='latin1', index=False)
y_train_sequential.to_csv("/content/drive/MyDrive/MSc project/y_train_sequential.csv", encoding ='latin1', index=False)

X_test_sequential.to_csv("/content/drive/MyDrive/MSc project/X_test_sequential.csv", encoding ='latin1', index=False)
y_test_sequential.to_csv("/content/drive/MyDrive/MSc project/y_test_sequential.csv", encoding ='latin1', index=False)


X_valid_sequential.to_csv("/content/drive/MyDrive/MSc project/X_valid_sequential.csv", encoding ='latin1', index=False)
y_valid_sequential.to_csv("/content/drive/MyDrive/MSc project/y_valid_sequential.csv", encoding ='latin1', index=False)
